In [245]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import pythoncom
import win32com.client as win32
import time

In [231]:
PATH = Path.cwd()
CORE_PATH = PATH.parent.parent.parent.parent.parent
CONFIG_PATH = PATH / 'configs'

PROFILING_PATH = CORE_PATH / 'Export_Data'
TARGET_PATH = CORE_PATH / 'HCP_Data'
if not PROFILING_PATH.exists():
    raise FileNotFoundError(f"Folder {PROFILING_PATH} không tồn tại! Vui lòng copy dữ liệu vào cùng folder với exe.")

In [ ]:
DF_PRO = pd.read_excel(PROFILING_PATH / 'Profiling_Pivot.xlsx', sheet_name= 'CHANNEL_SURVEY')
DF_TARGET = pd.read_excel(TARGET_PATH / 'Target_List.xlsx', sheet_name= 'TargetList')


In [279]:
DF_PRO['salesrep_name'] = DF_PRO['salesrep_name'].str.title().apply(lambda x: x.strip())

In [280]:
DF_TARGET['SALESREP_NAME'] = DF_TARGET['SALESREP_NAME'].str.title().apply(lambda x: x.strip())

In [281]:
DF_EMAIL = pd.read_excel(CONFIG_PATH / 'email_list.xlsx')
# Gộp email theo Bline thành chuỗi: "email1@domain.com; email2@domain.com"
email_mapping = DF_EMAIL.groupby('bline')['email'].apply(lambda x: '; '.join(x)).to_dict()

In [282]:
email_mapping

{'Alliance 1': 'khoa.dang1.nguyen@dksh.com',
 'Alliance 3': 'hec.comex.1@dksh.com; khoa.dang1.nguyen@dksh.com'}

In [283]:
with open(CONFIG_PATH / 'email_template.txt', 'r', encoding='utf-8') as f:
    template_content = f.read()

In [284]:
DF_TARGET = DF_TARGET[(DF_TARGET['cont_code'] != 'VN00000000')]
DF_TARGET = DF_TARGET[~DF_TARGET['Source.Name'].str.contains('TGTS', na=False)]

DF_TARGET['Bline'] = np.where(DF_TARGET['Source.Name'].str.contains('HA5'), 'Alliance 5',
        np.where(DF_TARGET['Source.Name'].str.contains('HECA 3'), 'Alliance 3',
        np.where(DF_TARGET['Source.Name'].str.contains('HECA 1'), 'Alliance 1',
        np.where(DF_TARGET['Source.Name'].str.contains('HECA6'), 'Alliance 6',
        np.where(DF_TARGET['Source.Name'].str.contains('OB'), 'OWN BRAND',
        np.where(DF_TARGET['Source.Name'].str.contains('HA8'), 'Alliance 8',DF_TARGET['Source.Name']))))))

DF_PRO_CHECK = DF_PRO[['salesrep_code','salesrep_name','cust_code','cont_code']]
DF_TARGET_CHECK = DF_TARGET[['Bline','FLM','SALESREP_NAME','PDA code', 'CustCode', 'cont_code']].drop_duplicates()
DF_TARGET_CHECK = DF_TARGET_CHECK.rename(columns={"PDA code":'salesrep_code'})

DF_PRO_SUM = DF_PRO_CHECK.groupby(['salesrep_code', 'salesrep_name'])['cont_code'].count().to_frame(name ='total_profiling').reset_index()
DF_TARGET_SUM = DF_TARGET_CHECK.groupby(['Bline', 'FLM','salesrep_code','SALESREP_NAME'])['cont_code'].nunique().to_frame(name ='total_target').reset_index()
CHECK = DF_TARGET_SUM.merge(DF_PRO_SUM[['salesrep_code','total_profiling']], how = 'left', on = 'salesrep_code')
CHECK['total_profiling'] = CHECK['total_profiling'].fillna(0)
CHECK['salesrep_code'] = CHECK['salesrep_code'].astype(str)

In [285]:
dfs={}
BLINES = CHECK['Bline'].unique().tolist()
for bline in BLINES:
    dfs[bline] = CHECK[CHECK['Bline'] == bline].copy()

In [286]:
TOTAL_PROFILING = CHECK['total_profiling'].sum()
TOTAL_TARGET = CHECK['total_target'].sum()
PERCENTIVE = (TOTAL_PROFILING*100.00/TOTAL_TARGET).round(1)

In [292]:
# --- VÒNG LẶP CHÍNH ---
for bline, df_sub in dfs.items():
    bline_clean = str(bline).strip()
    
    # KIỂM TRA: Nếu Bline này không có email thì bỏ qua luôn để an toàn
    if bline_clean not in email_mapping:
        print(f"⚠️ Cảnh báo: Không tìm thấy email cho Bline '{bline_clean}'. Bỏ qua không gửi.")
        continue
    
    target_emails = email_mapping[bline_clean]

    # --- BƯỚC 1: TÍNH TOÁN DỮ LIỆU ---
    # Tính tổng cho hàng Total
    sums = df_sub[['total_target', 'total_profiling']].sum()
    total_row = pd.DataFrame([sums])
    
    # Tính Achievement Rate cho hàng Total
    total_row['achievement_rate'] = ((total_row['total_profiling'] * 100.0) / total_row['total_target']).round(2)
    total_row['Bline'] = f"Total {bline_clean}"
    
    # Tính Achievement Rate cho các hàng chi tiết
    df_sub['achievement_rate'] = ((df_sub['total_profiling'] * 100.0) / df_sub['total_target']).round(2)
    
    # Gộp hàng tổng vào bảng chính
    total_data = pd.concat([df_sub, total_row], ignore_index=True).fillna('')
    total_data['achievement_rate'] = total_data['achievement_rate'].astype(str) + '%'

    # --- BƯỚC 2: TẠO VÀ FORMAT HTML TABLE ---
    widths = [120, 160, 120, 240, 120, 120, 160]
    html_table = total_data.to_html(index=False, border=1)

    # Thêm Style cho bảng
    html_table = html_table.replace(
        '<table border="1" class="dataframe">', 
        '<table style="border-collapse: collapse; font-family: Arial; font-size: 13px; table-layout: fixed; width: 1040px; border: 1px solid #ccc;">'
    )

    # Format Header (TH) với màu sắc và độ rộng
    th_matches = re.findall(r'<th>.*?</th>', html_table)
    for i, match in enumerate(th_matches):
        w = widths[i] if i < len(widths) else 100
        new_th = f'<th style="background-color: #0070C0; color: white; text-align: center; height: 30px; border: 1px solid #ccc; width: {w}px;">{match[4:-5]}</th>'
        html_table = html_table.replace(match, new_th, 1)

    # Format Cell (TD)
    # Tăng padding-left lên 15px để nội dung không dính vách
    cell_style = 'style="padding: 5px 5px 5px 15px; border: 1px solid #ccc; height: 25px; vertical-align: middle; text-align: left; word-wrap: break-word;"'
    html_table = html_table.replace('<td>', f'<td {cell_style}>')

    # --- BƯỚC 3: RÁP VÀO TEMPLATE VÀ GỬI MAIL ---
    # Giả sử các biến PERCENTIVE, TOTAL_PROFILING... được tính toán dựa trên df_sub của Bline đó
    data_to_fill = {
        "email_name": "Team Leader",
        "percentive": f"{(PERCENTIVE):.2f}%", 
        "total_profiling": f"{int(TOTAL_PROFILING):,}",
        "total_target": f"{int(TOTAL_TARGET):,}",
        "content": "---TABLE_PLACEHOLDER---"
    }

    # Format nội dung email
    email_body = template_content.format(**data_to_fill).replace('\n', '<br>')
    final_html_body = email_body.replace('---TABLE_PLACEHOLDER---', html_table)

    # Khởi tạo và gửi qua Outlook
    try:
        outlook = win32.Dispatch('outlook.application')
        mail = outlook.CreateItem(0)
        mail.Subject = f"[ComEx-Automation][Important] Nhờ hỗ trợ xúc tiến hoàn thành Profiling Survey - {bline_clean}"
        
        # ĐỊA CHỈ NGƯỜI NHẬN CHUẨN THEO BLINE
        mail.To = target_emails 
        
        mail.HTMLBody = f"<html><body>{final_html_body}</body></html>"
        
        # GỬI LUÔN
        mail.Send()
        print(f"✅ Đã gửi thành công Bline: {bline_clean} tới {target_emails}")
        
        # Nghỉ 1s để tránh Outlook bị nghẽn
        time.sleep(1)
        
    except Exception as e:
        print(f"❌ Lỗi khi gửi Bline {bline_clean}: {e}")

print("\n--- HOÀN THÀNH TẤT CẢ ---")

✅ Đã gửi thành công Bline: Alliance 1 tới khoa.dang1.nguyen@dksh.com
✅ Đã gửi thành công Bline: Alliance 3 tới hec.comex.1@dksh.com; khoa.dang1.nguyen@dksh.com
⚠️ Cảnh báo: Không tìm thấy email cho Bline 'Alliance 5'. Bỏ qua không gửi.
⚠️ Cảnh báo: Không tìm thấy email cho Bline 'Alliance 6'. Bỏ qua không gửi.
⚠️ Cảnh báo: Không tìm thấy email cho Bline 'Alliance 8'. Bỏ qua không gửi.
⚠️ Cảnh báo: Không tìm thấy email cho Bline 'OWN BRAND'. Bỏ qua không gửi.

--- HOÀN THÀNH TẤT CẢ ---
